# Laboratorio 7 — Regresión Logística
**SmartStay Advisors — Airbnb listings**

Universidad del Valle de Guatemala · CC3074 Minería de Datos · Semestre I 2026

## Inciso 1 — Variables dicotómicas por categoría de precio

Se pide construir **tres variables dicotómicas (0/1)** a partir de la variable respuesta categórica `price_category` creada en los laboratorios anteriores:

- `es_cara` → 1 si la vivienda es *cara*, 0 en cualquier otro caso.
- `es_media` → 1 si la vivienda es de precio *medio*, 0 en cualquier otro caso.
- `es_economica` → 1 si la vivienda es *económica*, 0 en cualquier otro caso.

Para mantener la reproducibilidad respecto a laboratorios previos se reutilizan los **mismos cortes por tercios (q1 = 33%, q2 = 66%)** sobre `price_num` y las mismas etiquetas usadas en el lab anterior.

In [1]:
# Librerías base para carga de datos y manipulación tabular
import pyreadr
import pandas as pd
import numpy as np

# Semilla fija para reproducibilidad en pasos posteriores del laboratorio
SEED = 42
np.random.seed(SEED)

In [2]:
# Lectura del RData original de listings (mismo archivo usado en labs anteriores)
result = pyreadr.read_r("listings.RData")
df = result["listings"].copy()

# Tamaño original del dataset antes de cualquier limpieza
print(f"Filas y columnas del dataset original: {df.shape}")

Filas y columnas del dataset original: (171748, 80)


In [3]:
# Normalización de la columna `price` a numérico, replicando la limpieza
# ya validada en los laboratorios previos: se remueven símbolos de moneda y
# separadores de miles, se tratan strings vacíos como nulos y se descartan
# filas sin precio (no se pueden clasificar sin la variable objetivo).
filas_antes = len(df)

df["price_num"] = (
    df["price"].astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)
df["price_num"] = df["price_num"].replace({"nan": np.nan, "None": np.nan, "": np.nan})
df["price_num"] = pd.to_numeric(df["price_num"], errors="coerce")
df = df.dropna(subset=["price_num"]).copy()

print(f"Filas antes de limpiar precio: {filas_antes:,}")
print(f"Filas con precio válido:       {len(df):,}")

Filas antes de limpiar precio: 171,748
Filas con precio válido:       76,246


In [4]:
# Cortes por tercios sobre price_num: mismos umbrales usados en el lab anterior
# para construir la variable categórica de precio. Se usan tercios (33%-66%)
# para obtener tres grupos balanceados en cantidad de observaciones.
q1, q2 = df["price_num"].quantile([1/3, 2/3])
print(f"q1 (percentil 33) = {q1:.2f}")
print(f"q2 (percentil 66) = {q2:.2f}")

# Construcción de la variable categórica de precio (3 niveles)
df["price_category"] = pd.cut(
    df["price_num"],
    bins=[-np.inf, q1, q2, np.inf],
    labels=["economica", "media", "cara"],
    include_lowest=True,
)

# Conteo por categoría para verificar balance
print("\nDistribución de price_category:")
print(df["price_category"].value_counts())

q1 (percentil 33) = 143.00
q2 (percentil 66) = 268.00

Distribución de price_category:
price_category
economica    25689
cara         25404
media        25153
Name: count, dtype: int64


In [5]:
# Creación de las tres variables dicotómicas pedidas en el inciso 1.
# Cada una es 1 cuando la observación pertenece a esa categoría, 0 en caso contrario.
df["es_cara"]      = (df["price_category"] == "cara").astype(int)
df["es_media"]     = (df["price_category"] == "media").astype(int)
df["es_economica"] = (df["price_category"] == "economica").astype(int)

# Resumen: totales y proporción de 1s por cada variable dicotómica
resumen = pd.DataFrame({
    "total_1s":  df[["es_cara", "es_media", "es_economica"]].sum(),
    "proporcion": df[["es_cara", "es_media", "es_economica"]].mean().round(4),
})
print(resumen)

# Chequeo de integridad: las tres variables deben ser mutuamente excluyentes
# y colectivamente exhaustivas, es decir, la suma por fila siempre debe ser 1.
suma_filas_ok = (df[["es_cara", "es_media", "es_economica"]].sum(axis=1) == 1).all()
print(f"\n¿Cada fila pertenece a exactamente 1 categoría? {suma_filas_ok}")

# Vista previa de las columnas generadas junto al precio y la categoría
df[["price_num", "price_category", "es_cara", "es_media", "es_economica"]].head(10)

              total_1s  proporcion
es_cara          25404      0.3332
es_media         25153      0.3299
es_economica     25689      0.3369

¿Cada fila pertenece a exactamente 1 categoría? True


,price_num,price_category,es_cara,es_media,es_economica
0,97.0,economica,0,0,1
1,160.0,media,0,1,0
2,38.0,economica,0,0,1
3,145.0,media,0,1,0
4,58.0,economica,0,0,1
5,49.0,economica,0,0,1
6,300.0,cara,1,0,0
7,150.0,media,0,1,0
8,165.0,media,0,1,0
9,117.0,economica,0,0,1


### Observaciones y conclusiones del inciso 1

- **Tamaño efectivo del dataset.** Del total original de `171,748` listings se conservan `76,246` filas tras descartar las que no tenían precio válido. Estas son las observaciones sobre las que se construirá toda la modelación del laboratorio.
- **Umbrales de tercios.** Los cortes calculados son `q1 ≈ 143` y `q2 ≈ 268`. Esto define:
  - *económica*: `price_num ≤ 143`
  - *media*: `143 < price_num ≤ 268`
  - *cara*: `price_num > 268`
- **Balance de clases.** La distribución resultante es prácticamente uniforme (≈33% en cada categoría: `economica = 25,689`, `media = 25,153`, `cara = 25,404`). Este balance es positivo porque los modelos de clasificación (en particular la regresión logística) no requerirán técnicas de re-muestreo para corregir desbalance, y las métricas como *accuracy* serán interpretables sin sesgo por clase mayoritaria.
- **Variables dicotómicas creadas.** Se generaron `es_cara`, `es_media` y `es_economica`, cada una con valores `{0, 1}`. Las tres son **mutuamente excluyentes y colectivamente exhaustivas** (la suma por fila es siempre 1), lo cual se verificó programáticamente.
- **Uso previsto.** La variable `es_cara` será la variable respuesta del modelo de regresión logística del inciso 3 (clasificación binaria *cara / no cara*), mientras que `es_media` y `es_economica` quedan disponibles para repetir el mismo flujo sobre las otras dos categorías si se desea comparar resultados.
- **Reproducibilidad.** Como los umbrales `q1` y `q2` se derivan de cuantiles fijos sobre el mismo archivo `listings.RData` y la limpieza de precio es determinística, la construcción de estas tres dicotómicas es totalmente reproducible entre ejecuciones y es consistente con la categorización usada en los laboratorios 5 y 6.